# SILAGEGUARD AI — Sensor AI Model Training Notebook
### SIH26111 — Smart AI-Enabled Rapid Feed and Silage Quality Testing System

This notebook trains a **Random Forest Classifier** to assess silage fermentation safety based on physical telemetry:
- **pH**: Lactic acid buffering and fermentation completeness
- **Moisture %**: Dry matter compaction and butyric/clostridial risk
- **Temperature & Ambient**: Aerobic respiration and microbial heat rise

The model is exported to a pure JSON decision tree structure for **zero-dependency on-device mobile execution**.

In [1]:
import os
import sys
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score, f1_score

print('Libraries imported successfully.')

## 1. Load Harmonized Agricultural Research Dataset

In [2]:
data_path = os.path.join('..', 'datasets', 'sensor', 'combined_silage_dataset.csv')
df = pd.read_csv(data_path)
print('Dataset Shape:', df.shape)
df.head()

## 2. Feature Engineering & Agronomic Transformation

In [3]:
OPTIMAL_PH = 4.0
OPTIMAL_MOISTURE = 64.0

df['delta_temp'] = df['temperature'] - df['ambient']
df['ph_deviation'] = (df['ph'] - OPTIMAL_PH).abs()
df['moisture_deviation'] = (df['moisture'] - OPTIMAL_MOISTURE).abs()
df['temp_rise'] = np.maximum(0.0, df['delta_temp'])

feature_cols = ['ph', 'moisture', 'temperature', 'ambient', 'delta_temp', 'ph_deviation', 'moisture_deviation', 'temp_rise']
label_map = {'Safe': 0, 'Caution': 1, 'Unsafe': 2}

X = df[feature_cols]
y = df['label'].map(label_map).values
print('Feature matrix shape:', X.shape)

## 3. Model Training & Cross Validation (Random Forest vs Baseline)

In [4]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.20, random_state=42, stratify=y)

rf = RandomForestClassifier(n_estimators=25, max_depth=8, random_state=42, class_weight='balanced')
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
scores = cross_val_score(rf, X_train, y_train, cv=cv, scoring='f1_macro')
print(f'5-Fold Stratified Macro F1: {scores.mean():.4f} (+/- {scores.std():.4f})')

rf.fit(X_train, y_train)
y_pred = rf.predict(X_test)
print('Test Accuracy:', accuracy_score(y_test, y_pred))
print('Test Macro F1:', f1_score(y_test, y_pred, average='macro'))

## 4. Feature Importance & Classification Report

In [5]:
print(classification_report(y_test, y_pred, target_names=['Safe', 'Caution', 'Unsafe']))

importances = pd.Series(rf.feature_importances_, index=feature_cols).sort_values(ascending=False)
print('Feature Importances:')
print(importances)

## 5. Export to On-Device JSON Tree

In [6]:
sys.path.append(os.path.join('..', 'sensor_model'))
from export_rf_json import export_random_forest_to_json

out_json = os.path.join('..', 'mobile', 'assets', 'models', 'sensor_rf_model.json')
export_random_forest_to_json(rf, feature_cols, ['Safe', 'Caution', 'Unsafe'], out_json)
print('Model successfully exported to mobile assets.')